# 2. Statistisk Analys
Deskriptiv statistik, fördelningar och normalitetstest för WHR26-variablerna.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("../data/cleaned_data.csv")

In [ ]:
# De 6 lyckofaktorer som WHR använder för att förklara ett lands lyckopoäng.
# Dessa kolumnnamn matchar exakt cleaned_data.csv och används i samtliga analyser nedan.
faktorer = [
    "Explained by: Log GDP per capita",
    "Explained by: Social support",
    "Explained by: Healthy life expectancy",
    "Explained by: Freedom to make life choices",
    "Explained by: Generosity",
    "Explained by: Perceptions of corruption"
]

# Skriv ut antal rader och unika länder i datasetet. 
# Förväntad output: 975 rader och 141 unika länder.
print(f"Rader: {df.shape[0]}, Länder: {df['Country name'].nunique()}")

In [ ]:
# Beräkna förändringen i "Life evaluation (3-year average)" mellan 2019 och 2025 för varje land
# Skapa två DataFrames, en för 2019 och en för 2025, och slå ihop dem på "Country name"
df_2019 = df[df["Year"] == 2019].set_index("Country name")[["Life evaluation (3-year average)"]]
df_2025 = df[df["Year"] == 2025].set_index("Country name")[["Life evaluation (3-year average)"]]

df_förändring = df_2019.join(df_2025, lsuffix="_2019", rsuffix="_2025")
df_förändring["delta"] = (
    df_förändring["Life evaluation (3-year average)_2025"]
    - df_förändring["Life evaluation (3-year average)_2019"]
)
df_förändring = df_förändring.sort_values("delta", ascending=False)

print(df_förändring.head(10))   # topp 10 vinnare
print(df_förändring.tail(10))   # topp 10 förlorare

In [ ]:
# Visa topp 10 vinnare och förlorare
# Vi kan runda av "delta" till 3 decimaler för bättre läsbarhet
topp10_vinnare   = df_förändring.head(10)
topp10_förlorare = df_förändring.tail(10)

print("=== VINNARE ===")
print(topp10_vinnare[["delta"]].round(3))

print("\n=== FÖRLORARE ===")
print(topp10_förlorare[["delta"]].round(3))

In [ ]:
# Skapa en korrelationsmatris för de 6 lyckofaktorerna
# Visar hur starkt de 6 faktorerna hänger ihop med varandra. 
# Celler nära +1 (röd) = rör sig tillsammans, nära 0 (vit) = oberoende, nära −1 (blå) = omvänt samband.
korr = df[faktorer].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(
    korr,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    vmin=-1, vmax=1,
    square=True
)
# Vi kan lägga till en titel och justera layouten för att göra det snyggare
plt.title("Korrelationsmatris – de 6 lyckofaktorerna")
plt.tight_layout()
plt.show()

In [ ]:
# Visar genomsnittsvärdet på varje faktor för de 10 vinnarna respektive de 10 förlorarna (vid 2019 — startläget). 
# Raden med störst skillnad är den faktor som troligtvis skiljer dem åt mest.
df_2019_full = df[df["Year"] == 2019].set_index("Country name")

# Beräkna genomsnittet av varje faktor för topp 10 vinnare och topp 10 förlorare
vinnare_snitt = df_2019_full.loc[topp10_vinnare.index, faktorer].mean()
förlorare_snitt = df_2019_full.loc[topp10_förlorare.index, faktorer].mean()

# Skapa en DataFrame för att jämföra vinnare och förlorare
jämförelse = pd.DataFrame({
    "Vinnare (snitt)": vinnare_snitt,
    "Förlorare (snitt)": förlorare_snitt,
    "Skillnad": vinnare_snitt - förlorare_snitt
}).round(3)

print(jämförelse.sort_values("Skillnad", ascending=False))